# Day 036 Solution — Pandas Fundamentals

End-to-end analysis: load a sales dataset, inspect it, add computed columns, filter, and aggregate.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import warnings
warnings.filterwarnings('ignore')
import pandas as pd

def summarize_dataframe(df: pd.DataFrame) -> dict:
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    null_count   = int(df.isnull().sum().sum())
    return {
        'rows':         df.shape[0],
        'cols':         df.shape[1],
        'columns':      df.columns.tolist(),
        'numeric_cols': numeric_cols,
        'null_count':   null_count,
    }


import io
import pandas as pd

def load_csv(csv_text: str) -> pd.DataFrame:
    return pd.read_csv(io.StringIO(csv_text))


import pandas as pd

def filter_by_threshold(df: pd.DataFrame, column: str, threshold: float) -> pd.DataFrame:
    mask = df[column] >= threshold
    return df[mask].reset_index(drop=True)


import pandas as pd

def add_category_column(df: pd.DataFrame, price_col: str) -> pd.DataFrame:
    def categorize(price):
        if price < 20:
            return 'budget'
        elif price < 100:
            return 'standard'
        else:
            return 'premium'
    result = df.copy()
    result['category'] = result[price_col].apply(categorize)
    return result


import pandas as pd

def group_summary(df: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    summary = (
        df.groupby(group_col)[value_col]
        .agg(total='sum', count='count')
        .reset_index()
        .sort_values('total', ascending=False)
        .reset_index(drop=True)
    )
    return summary

## Step 1 — Load & Inspect

In [ ]:
SALES_CSV = (
    'order_id,product,unit_price,quantity\n'
    '1,Widget,25,10\n'
    '2,Gadget,150,3\n'
    '3,Widget,25,20\n'
    '4,Doohickey,8,50\n'
    '5,Gadget,150,5\n'
    '6,Thingamajig,200,2\n'
    '7,Widget,25,4\n'
    '8,Doohickey,8,15\n'
    '9,Thingamajig,200,8\n'
    '10,Gadget,150,1'
)

df = load_csv(SALES_CSV)
info = summarize_dataframe(df)
print('Shape:', df.shape)
print('Columns:', info['columns'])
print('Numeric cols:', info['numeric_cols'])
print('Null count:', info['null_count'])
print(df.head())

assert df.shape == (10, 4)
assert info['null_count'] == 0

## Step 2 — Add Revenue Column

In [ ]:
df['revenue'] = df['unit_price'] * df['quantity']
print(df[['product', 'unit_price', 'quantity', 'revenue']].to_string(index=False))

assert 'revenue' in df.columns
assert df['revenue'].sum() == (
    25*10 + 150*3 + 25*20 + 8*50 + 150*5 + 200*2 + 25*4 + 8*15 + 200*8 + 150*1
)

## Step 3 — Add Price Category

In [ ]:
df = add_category_column(df, 'unit_price')
print(df[['product', 'unit_price', 'category']].drop_duplicates().to_string(index=False))

assert 'category' in df.columns
# Doohickey 8 -> budget, Widget 25 -> standard,
# Gadget 150 -> premium, Thingamajig 200 -> premium
cats = dict(zip(df['product'], df['category']))
assert cats['Doohickey']    == 'budget'
assert cats['Widget']       == 'standard'
assert cats['Gadget']       == 'premium'
assert cats['Thingamajig']  == 'premium'

## Step 4 — Filter High-Value Orders

In [ ]:
high_value = filter_by_threshold(df, 'revenue', 500)
print(f'{len(high_value)} orders with revenue >= 500:')
print(high_value[['order_id', 'product', 'revenue']].to_string(index=False))

assert (high_value['revenue'] >= 500).all()
assert len(high_value) > 0

## Step 5 — Group Summary by Product

In [ ]:
product_summary = group_summary(df, 'product', 'revenue')
print('Revenue by product (sorted):')
print(product_summary.to_string(index=False))

assert list(product_summary.columns) == ['product', 'total', 'count']
assert product_summary.iloc[0]['total'] >= product_summary.iloc[1]['total']

## Step 6 — Assemble Report

In [ ]:
report = {
    'total_revenue':     df['revenue'].sum(),
    'top_product':       product_summary.iloc[0]['product'],
    'avg_order_value':   round(df['revenue'].mean(), 2),
    'category_breakdown': group_summary(df, 'category', 'revenue'),
}

print(f"Total revenue    : {report['total_revenue']}")
print(f"Top product      : {report['top_product']}")
print(f"Avg order value  : {report['avg_order_value']}")
print('Category breakdown:')
print(report['category_breakdown'].to_string(index=False))

assert report['total_revenue'] > 0
assert isinstance(report['top_product'], str)
assert report['avg_order_value'] > 0

print('\nPandas Fundamentals complete!')